# Fine-tune TinyLlama with a DesiData Q&A Dataset

This notebook downloads a public DesiData CSV, turns question-and-answer rows into instruction examples, and fine-tunes `TinyLlama/TinyLlama-1.1B-Chat-v1.0` with LoRA.

Run this in Google Colab with a T4 GPU, or on a local CUDA-enabled machine. Replace `DATASET_URL` with the public download URL from any DesiData dataset page.

In [ ]:
# Colab: Runtime -> Change runtime type -> T4 GPU, then run this cell.
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes pandas

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_URL = "https://www.desidata.in/api/datasets/national-health-profile-2022-question-and-answer-dataset/download"
OUTPUT_DIR = "./tinyllama-desidata-lora"

assert torch.cuda.is_available(), "Use a CUDA GPU runtime (for example, Colab T4) to run this notebook."
print("GPU:", torch.cuda.get_device_name(0))

## Load and inspect the dataset

The URL is public and works directly with pandas. This example expects `question` and `answer` columns. Change the column names below if your CSV uses different ones.

In [ ]:
df = pd.read_csv(DATASET_URL)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
display(df.head())
print(df.columns.tolist())

In [ ]:
QUESTION_COLUMN = "question"
ANSWER_COLUMN = "answer"

required_columns = {QUESTION_COLUMN, ANSWER_COLUMN}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}. Update QUESTION_COLUMN and ANSWER_COLUMN."

training_df = df[[QUESTION_COLUMN, ANSWER_COLUMN]].dropna().copy()
training_df[QUESTION_COLUMN] = training_df[QUESTION_COLUMN].astype(str).str.strip()
training_df[ANSWER_COLUMN] = training_df[ANSWER_COLUMN].astype(str).str.strip()
training_df = training_df[(training_df[QUESTION_COLUMN] != "") & (training_df[ANSWER_COLUMN] != "")]

print(f"Training examples: {len(training_df):,}")
display(training_df.head())

## Format examples for TinyLlama chat

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def format_example(row):
    messages = [
        {"role": "system", "content": "You are a helpful assistant answering questions using Indian public data."},
        {"role": "user", "content": row[QUESTION_COLUMN]},
        {"role": "assistant", "content": row[ANSWER_COLUMN]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

train_dataset = Dataset.from_pandas(training_df, preserve_index=False)
train_dataset = train_dataset.map(lambda row: {"text": format_example(row)})
print(train_dataset[0]["text"])

## Load TinyLlama in 4-bit and configure LoRA

LoRA trains a small adapter instead of all model weights, making this feasible on a Colab T4 GPU.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

## Train

Start with one epoch to validate the pipeline. Increase `num_train_epochs` only after reviewing outputs and dataset quality.

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_seq_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

## Test the fine-tuned adapter

In [ ]:
prompt = "What is the dataset about?"
messages = [
    {"role": "system", "content": "You are a helpful assistant answering questions using Indian public data."},
    {"role": "user", "content": prompt},
]

input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=160,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

response_ids = output_ids[0][input_ids.shape[-1]:]
print(tokenizer.decode(response_ids, skip_special_tokens=True))

## Next steps

- Evaluate answers against a held-out validation set before deployment.
- Keep licences and source citations with the model and its outputs.
- Upload the adapter directory to Hugging Face Hub only after validation.
- Use the DesiData dataset page for the public URL, preview, documentation, and download details.